# CS604 Colab launcher — Ana: paired comparison of the controlled PEFT arms (inference only)

Are the differences in the controlled PEFT table (`colab_peft_336px_controlled-comparison_all-10000-classes.ipynb`) real? Every arm was scored on the **same 50,000 holdout photos** (all 10,000 classes, 5 per class, subset seed 0), so they can be compared photo by photo with `scripts/paired_compare.py`:

- fixed (A wrong, B right), broken (A right, B wrong), both wrong;
- the difference B − A with a 95% interval from a paired bootstrap over **classes** (the honest one) and over photos;
- McNemar's exact test (are fixes and breaks equally likely?);
- are the fixes siblings? (A's wrong guess within 5 class IDs of the truth, as in `rerank_knn.py`).

Comparisons: every arm against **A (LoRA r24)**, including A_seed1/2, and **C (AdaptFormer)** against C_seed1/2. The seed pairs measure how far two runs of the *same* method differ, which the bootstrap interval does not cover. A difference is called **real** only if its class interval excludes 0 *and* it is larger than the biggest seed-pair difference.

No training. Arms that have not finished yet are skipped; run again after the second PEFT session to add them (each arm is scored once and cached on Drive). About 15 min for the data, then a few minutes per arm on an A100; any GPU works.

Outputs in `CS604_RUNS/paired_peft_controlled/`: `paired_summary_<baseline>.csv`, `paired_<A>_vs_<B>.json`, `changes_<A>_vs_<B>.csv` (every photo either model got wrong, for picking report examples), `preds_<arm>.pt` (cached predictions). Use **Runtime → Run all**.

In [ ]:
REPO_URL = "https://github.com/keaidewoq/cs604-project.git"
REPO_DIR = "/content/repo"
DATA_DIR = "/content/data"
# Must match a folder (or shortcut) in My Drive — not the browser URL.
DRIVE_OUT = "/content/drive/MyDrive/CS604_RUNS"
BRANCH = "ana-branch"
PILOT_DATA_DIR = "/content/pilot_data"
ABLATION = "ablation_ablation_peft_controlled"  # where the PEFT launcher wrote its arms
OUT_NAME = "paired_peft_controlled"
# baseline arm -> arms compared with it, all on the same holdout
COMPARISONS = {
    "A_lora_r24": ["A_seed1", "A_seed2", "B_lora_r40_blocks4_11", "C_adaptformer_w192", "D_adapter_w96",
                   "E_bitfit", "F_lastblock_attention", "G_prefix_p50"],
    "C_adaptformer_w192": ["C_seed1", "C_seed2"],
}
SEED_ARMS = {"A_seed1", "A_seed2", "C_seed1", "C_seed2"}
SEED_NOISE_FALLBACK = 0.3  # points; used until a seed arm has finished
# End the runtime after the last cell, or as soon as a step fails.
AUTO_DISCONNECT = True


def end_runtime(reason):
    """Flush Drive writes, then release the GPU so it stops using compute units."""
    print(reason, flush=True)
    if not AUTO_DISCONNECT:
        return
    from google.colab import drive, runtime
    try:
        drive.flush_and_unmount()  # without this the last checkpoint can be lost
        print("Drive flushed and unmounted", flush=True)
    except Exception as error:
        print(f"Drive flush failed: {error}", flush=True)
    runtime.unassign()

## 1. Clone or pull the repo

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub PAT (hidden; repo scope). Leave empty if the repo is public: ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    if token:
        !git -C {REPO_DIR} pull {clone_url} {BRANCH}
    else:
        !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {clone_url} {REPO_DIR}

%cd {REPO_DIR}
!git branch --show-current
!git rev-parse --short HEAD

## 2. Install requirements

In [ ]:
%pip install -r requirements.txt

## 3. Mount Drive and check `CS604_RUNS`

If this cell errors, add a shortcut named `CS604_RUNS` to My Drive from the [shared folder](https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR).

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

out = Path(DRIVE_OUT)
if not out.is_dir():
    raise FileNotFoundError(
        f"{DRIVE_OUT} not found after mounting.\n"
        "Open https://drive.google.com/drive/folders/107nc4GKekmaKe38GR_8VBn3Bh_xH-ZtR "
        "→ Organize → Add shortcut to Drive → My Drive, and name the shortcut CS604_RUNS."
    )
out.mkdir(parents=True, exist_ok=True)
print("Writing runs to", out.resolve())
print("Existing items:", [p.name for p in out.iterdir()][:20])

## 4. Download data to `/content/data`

In [ ]:
!python scripts/download_data.py --dest {DATA_DIR}

## 5. Rebuild the holdout split

The same seeded split the arms were trained on: 400,000 fit photos, 50,000 holdout photos. Takes seconds (text files and symlinks).

In [ ]:
from scripts.pilot_sweep import build_pilot_root

# Same split as the PEFT arms were trained with (configs/peft_336px_controlled-comparison_all-10000-classes.yaml: pilot)
PILOT_ROOT, info = build_pilot_root(DATA_DIR, PILOT_DATA_DIR, 10000, 5, 0)
print(PILOT_ROOT, info)

## 6. Find the finished arms

In [ ]:
import json
from pathlib import Path

arms_dir = Path(DRIVE_OUT) / ABLATION / "arms"
RUN_DIRS = {p.stem: json.loads(p.read_text())["run_dir"] for p in sorted(arms_dir.glob("*.json"))}
wanted = set(COMPARISONS) | {b for others in COMPARISONS.values() for b in others}
print("finished:", sorted(wanted & set(RUN_DIRS)))
print("not finished yet (skipped):", sorted(wanted - set(RUN_DIRS)))

## 7. Paired comparisons (inference only)

Scores each arm once on the holdout and caches `preds_<arm>.pt` on Drive; reruns only redo the statistics.

In [ ]:
import subprocess
import sys

OUT = Path(DRIVE_OUT) / OUT_NAME
for base, others in COMPARISONS.items():
    ready = [b for b in others if b in RUN_DIRS]
    if base not in RUN_DIRS or not ready:
        print(f"skipping baseline {base}: nothing to compare yet")
        continue
    cmd = [sys.executable, "-u", "scripts/paired_compare.py", "--pilot_root", str(PILOT_ROOT),
           "--out_dir", str(OUT), "--baseline", base]
    cmd += [arg for name in [base, *ready] for arg in ("--run", f"{name}={RUN_DIRS[name]}")]
    proc = subprocess.run(cmd, stdout=subprocess.DEVNULL)  # the JSON is shown in the next cell
    if proc.returncode != 0:
        end_runtime(f"paired comparison against {base} failed (exit {proc.returncode})")
        raise SystemExit("paired comparison failed")

## 8. Results

`B-A` and the intervals are in points. `ci95_class` resamples whole classes (report this one); `ci95_photo` resamples photos and is narrower. **Verdict**: *tie* if the class interval includes 0; *within seed spread* if it excludes 0 but the gap is no bigger than the largest seed-pair difference; *real* if both tests pass. If `sibling_share_fixes` is above `sibling_share_errors_A`, B fixes sibling confusions more than their share of A's errors.

In [ ]:
import pandas as pd

rows = [json.loads(p.read_text()) for p in sorted(OUT.glob("paired_*_vs_*.json"))]
seed_diffs = [abs(r["difference_points"]) for r in rows if r["b"] in SEED_ARMS]
seed_spread = max(seed_diffs) if seed_diffs else SEED_NOISE_FALLBACK
print(f"seed spread: {seed_spread:.2f} points "
      f"({'largest seed-pair difference' if seed_diffs else 'fallback, no seed arm yet'})")


def verdict(r):
    lo, hi = r["difference_ci95_by_class"]
    if r["b"] in SEED_ARMS:
        return "seed pair"
    if lo <= 0 <= hi:
        return "tie"
    return "real" if abs(r["difference_points"]) > seed_spread else "within seed spread"


table = pd.DataFrame([{
    "A": r["a"], "B": r["b"], "top1_A": r["accuracy_a"]["top1"], "top1_B": r["accuracy_b"]["top1"],
    "B-A": r["difference_points"], "ci95_class": r["difference_ci95_by_class"],
    "ci95_photo": r["difference_ci95_by_photo"], "fixed": r["fixed"], "broken": r["broken"],
    "mcnemar_p": r["mcnemar_p"],
    "sibling_share_errors_A": r["by_distance"]["sibling_share_of_errors"],
    "sibling_share_fixes": r["by_distance"]["sibling_share_of_fixes"],
    "verdict": verdict(r)} for r in rows])
display(table)

## 9. End the runtime

In [ ]:
end_runtime("all cells finished; ending the runtime")